### This table would contain the total sales for each policy type and each month. It would be used to analyze the performance of different policy types over time.

### This table would contain the number and amount of claims by policy type and claim status. It would be used to monitor the claims process and identify any trends or issues.

In [0]:
%sql
create or replace temp view vw_gold_claims_by_policy_type_and_status 
AS
SELECT
    policy_type,
    claim_status,
    COUNT(*) AS total_claims,
    SUM(claim_amount) AS total_claim_amount
FROM
    silverlayer.claim c
JOIN silverlayer.policy p
    ON c.policy_id = p.policy_id
GROUP BY
    policy_type,
    claim_status
HAVING p.policy_type IS NOT NULL;

In [0]:
%sql
select * from vw_gold_claims_by_policy_type_and_status 

### Merge into gold layer

In [0]:
%sql
Merge into goldlayer.claims_by_policy_type_and_status AS T
USING vw_gold_claims_by_policy_type_and_status AS S
ON t.policy_type = s.policy_type
and t.claim_status = s.claim_status

WHEN MATCHED THEN
UPDATE
SET
    T.total_claim_amount = s.total_claim_amount,
    T.total_claims = s.total_claims,
    T.updated_timestamp = current_timestamp()

WHEN NOT MATCHED THEN
INSERT (
    policy_type,
    claim_status,
    total_claim_amount,
    total_claims,
    updated_timestamp
)
VALUES (
    s.policy_type,
    s.claim_status,
    s.total_claim_amount,
    s.total_claims,
    current_timestamp()
);

In [0]:
%sql
select * from goldlayer.claims_by_policy_type_and_status

### Analyze the claim data based on the policy type like AVG, MAX, MIN, Count of claim.

In [0]:
%sql
create or replace temp view vw_gold_claims_analysis as
SELECT
    policy_type,
    AVG(claim_amount) AS avg_claim_amount,
    MAX(claim_amount) AS max_claim_amount,
    MIN(claim_amount) AS min_claim_amount,
    COUNT(DISTINCT claim_id) AS total_claims
FROM
    silverlayer.claim c
JOIN silverlayer.policy p
    ON c.policy_id = p.policy_id
GROUP BY policy_type
HAVING p.policy_type IS NOT NULL;

In [0]:
%sql
select * from vw_gold_claims_analysis

In [0]:
%sql
Merge into goldlayer.claims_analysis AS T USING vw_gold_claims_analysis AS S ON t.policy_type = s.policy_type
 WHEN MATCHED THEN
UPDATE
SET
    T.avg_claim_amount = s.avg_claim_amount,T.max_claim_amount = s.max_claim_amount,
    T.min_claim_amount = s.min_claim_amount,T.total_claims = s.total_claims,T.updated_timestamp = current_timestamp()
WHEN NOT MATCHED THEN
INSERT (
    policy_type,avg_claim_amount,max_claim_amount,min_claim_amount,total_claims,updated_timestamp
)
VALUES (
    s.policy_type,
    s.avg_claim_amount,
    s.max_claim_amount,
    s.min_claim_amount,
    s.total_claims,
    current_timestamp()
);

In [0]:
%sql
select * from goldlayer.claims_analysis